# Multitask training — GS40 + GS55 + GS33 combined

Train **one** ConvNeXt + StarDist multitask model on tiles from all three gestational ages.

## How to use this notebook

1. **Edit cell 4 (`PARAMETERS`)** — paths, hyperparameters, backbone, run flags.
2. **Run all cells** — writes `configs/config_gs40_gs55_gs33_multitask.yaml` and the matching finetune config.
3. To launch training: either set `RUN_TRAINING = True` and `RUN_STAGE` in `PARAMETERS`, **or** copy-paste the terminal command printed by the last cell.

## Prerequisites
For each cohort: PNG tiles + instance masks + `*_inst2class.json` sidecars + split CSVs (`train.csv`, `val.csv`).
GS33 needs `make_training_dataset/GS33/2_build_dataset_gs33_256.ipynb` to have been run first.

## Where the logic lives
- Helper functions: `aux_codes/train_utils.py`
- Training entrypoint: `shared_convnext_stardist_decoder/train_v2.py` (`python -m shared_convnext_stardist_decoder.train_v2 --config <yaml>`)
- Configs written by this notebook: `shared_convnext_stardist_decoder/configs/`


In [ ]:
# ── Bootstrap + imports (run once) ────────────────────────────────────────
import sys
from pathlib import Path

_REPO_ROOT = Path.cwd().resolve().parents[1]   # notebook is in <repo>/shared_convnext_stardist_decoder/notebooks_train/
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

from shared_convnext_stardist_decoder.aux_codes.train_utils import (
    build_training_config, read_stems, run_training, write_config,
)

print("OK — train_utils loaded")


---
## PARAMETERS — edit only here

Every variable a typical experiment changes lives in this cell. Logic cells
below should not need editing for: swapping cohort paths, changing LR or epochs,
switching backbone (tiny → base → large), or toggling whether training runs in
the notebook vs from the terminal.


In [ ]:
# 1. COHORTS — add/remove a key to add/remove a dataset
COHORTS: dict[str, dict] = {
    "GS40": {
        "root":         Path(r"\\kittyserverdw\Andre_kit\data\students\Diogo\data\fetal\GS40\cellvit_training\dataset_256_40k_48_slides"),
        "train_images": "train/images",
        "train_labels": "stardist_multitask_ready/train_instance_labels",
        "val_images":   "train/images",   # GS40 val tiles share train folder; only stems differ
        "val_labels":   "stardist_multitask_ready/train_instance_labels",
        "train_split":  "splits/fold_0/train.csv",
        "val_split":    "splits/fold_0/val.csv",
    },
    "GS55": {
        "root":         Path(r"\\kittyserverdw\Andre_kit\data\students\Diogo\data\fetal\GS55\cellvit_training\dataset_256_50k_clsbal_GS55"),
        "train_images": "train/images",
        "train_labels": "train/labels",
        "val_images":   "val/images",
        "val_labels":   "val/labels",
        "train_split":  "splits/fold_0/train.csv",
        "val_split":    "splits/fold_0/val.csv",
    },
    "GS33": {
        "root":         Path(r"\\kittyserverdw\Andre_kit\data\students\Diogo\data\fetal\GS33\cellvit_training\dataset_256_30k_GS33"),
        "train_images": "train/images",
        "train_labels": "train/labels",
        "val_images":   "val/images",
        "val_labels":   "val/labels",
        "train_split":  "splits/fold_0/train.csv",
        "val_split":    "splits/fold_0/val.csv",
    },
}

# 2. Experiment metadata
EXPERIMENT_TAG = "gs40_gs55_gs33_v4"   # appears in experiment_name, config filename, ckpt dir
CKPT_PARENT    = COHORTS["GS40"]["root"] / "convnext_stardist_multitask_runs"

# 3. Backbone — swap to base or large by changing this string
BACKBONE = "facebook/convnextv2-tiny-22k-224"
# Alternatives:
#   "facebook/convnextv2-base-22k-224"   (~89M params,  ~1.6–1.9× slower inference)
#   "facebook/convnextv2-large-22k-224"  (~198M params, ~2.3–2.7× slower inference)

# 4. Hyperparameters per stage — same keys, two stages
HPARAMS: dict[str, dict] = {
    "main": dict(
        epochs=35, batch_size=8, lr=5e-5, lr_min=1e-6, weight_decay=0.03,
        freeze_backbone_epochs=10,
        loss_w_prob=2.0, loss_w_dist=0.15, loss_w_cls=1.0, loss_w_inst=0.25,
        cls_semantic_dim=128, num_workers=8,
    ),
    "finetune": dict(
        epochs=15, batch_size=8, lr=1e-5, lr_min=1e-7, weight_decay=0.03,
        freeze_backbone_epochs=5,
        loss_w_prob=2.0, loss_w_dist=0.15, loss_w_cls=1.0, loss_w_inst=0.25,
        cls_semantic_dim=128, num_workers=8,
    ),
}

# 5. Run-time flags
RUN_TRAINING      = False           # True = launch training in this notebook
RUN_STAGE         = "main"          # "main" | "finetune"  (which config to launch)
RESUME_CHECKPOINT = None            # Path to .pt for warm-start, else None
RESUME_STRICT     = True

print(f"Cohorts        : {list(COHORTS)}")
print(f"Experiment tag : {EXPERIMENT_TAG}")
print(f"Backbone       : {BACKBONE}")
print(f"Run training   : {RUN_TRAINING}  (stage={RUN_STAGE})")


---
## Data constants

Class taxonomy + repo paths. Rarely change.

> **Class naming bug** (do not fix until public release): the class named
> `spleen` (idx 7) is actually the **gonad**, and `spleen2` (idx 18) is the
> **spleen**. See `CLAUDE.md` for the rename plan.


In [ ]:
CLASS_NAMES = [
    "bone", "brain", "eye", "heart", "lungs", "gi", "liver", "spleen",
    "pancreas", "kidney", "mesokidney", "collagen", "ear", "nontissue",
    "thymus", "thyroid", "bladder", "skull", "spleen2",
]

CONFIG_DIR = _REPO_ROOT / "shared_convnext_stardist_decoder" / "configs"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
print(f"CONFIG_DIR : {CONFIG_DIR}")


---
## Load and merge stems

One loop over `COHORTS` produces the combined train/val stem lists.
Adding a cohort is a single `COHORTS[...]` key.


In [ ]:
def _load(p: Path, label: str) -> list[str]:
    if not p.is_file():
        raise FileNotFoundError(f"{label}: {p}")
    s = read_stems(p); print(f"  {label}: {len(s):,} stems"); return s

train_stems, val_stems = [], []
for tag, c in COHORTS.items():
    train_stems += _load(c["root"] / c["train_split"], f"{tag} train")
    val_stems   += _load(c["root"] / c["val_split"],   f"{tag} val")

train_stems = list(dict.fromkeys(train_stems))   # ordered dedupe
val_stems   = list(dict.fromkeys(val_stems))

overlap = set(train_stems) & set(val_stems)
print(f"\nCombined : train {len(train_stems):,} | val {len(val_stems):,}")
print(f"Train ∩ Val : {len(overlap)} (should be 0)")


---
## Write training and finetune configs

Single `make_config(stage)` helper, called twice. Both configs share the same
cohort sources and class taxonomy; only the hyperparameter block changes.


In [ ]:
def make_config(stage: str) -> Path:
    """Build and write the YAML config for one stage. Returns the written path."""
    hp          = HPARAMS[stage]
    suffix      = "" if stage == "main" else "_finetune"
    experiment  = f"convnext_stardist_mt_{EXPERIMENT_TAG}{suffix}"
    fname       = (f"config_{EXPERIMENT_TAG}_multitask.yaml" if stage == "main"
                   else f"config_finetune_{EXPERIMENT_TAG}.yaml")
    config_path = CONFIG_DIR / fname
    ckpt_out    = CKPT_PARENT / f"run_{EXPERIMENT_TAG}{suffix}"
    ckpt_out.mkdir(parents=True, exist_ok=True)

    # build_training_config() needs primary train_images / inst_labels for backward compat;
    # they get overwritten by train_sources/val_sources below for multi-root training.
    primary = next(iter(COHORTS.values()))
    cfg = build_training_config(
        experiment_name        = experiment,
        train_images_dir       = primary["root"] / primary["train_images"],
        inst_labels_dir        = primary["root"] / primary["train_labels"],
        ckpt_out               = ckpt_out,
        train_stems            = train_stems,
        val_stems              = val_stems,
        class_names            = CLASS_NAMES,
        epochs                 = hp["epochs"],
        batch_size             = hp["batch_size"],
        lr                     = hp["lr"],
        lr_min                 = hp["lr_min"],
        weight_decay           = hp["weight_decay"],
        freeze_backbone_epochs = hp["freeze_backbone_epochs"],
        loss_w_cls             = hp["loss_w_cls"],
        loss_w_inst            = hp["loss_w_inst"],
        loss_w_dist            = hp["loss_w_dist"],
        cls_semantic_dim       = hp["cls_semantic_dim"],
    )

    # Patches for keys not exposed by build_training_config():
    cfg["model"]["backbone"]    = BACKBONE
    cfg["train"]["loss_w_prob"] = hp["loss_w_prob"]
    cfg["train"]["num_workers"] = hp["num_workers"]
    cfg["data"]["train_sources"] = [
        {"images_dir": str(c["root"] / c["train_images"]),
         "labels_dir": str(c["root"] / c["train_labels"])} for c in COHORTS.values()
    ]
    cfg["data"]["val_sources"] = [
        {"images_dir": str(c["root"] / c["val_images"]),
         "labels_dir": str(c["root"] / c["val_labels"])} for c in COHORTS.values()
    ]

    write_config(cfg, config_path)
    return config_path


MAIN_CONFIG     = make_config("main")
FINETUNE_CONFIG = make_config("finetune")
print(f"\n  main     → {MAIN_CONFIG}")
print(f"  finetune → {FINETUNE_CONFIG}")


---
## Run training

`RUN_TRAINING=False` (default) writes configs only — launch training from the terminal:

```powershell
conda activate convnext_stardist_mt
cd C:\Users\Andre\cursor_projects\Convnext_stardist

# Fresh main run
python -m shared_convnext_stardist_decoder.train_v2 --config shared_convnext_stardist_decoder\configs\config_gs40_gs55_gs33_v4_multitask.yaml

# Finetune from a previous best.pt
python -m shared_convnext_stardist_decoder.train_v2 ^
    --config shared_convnext_stardist_decoder\configs\config_finetune_gs40_gs55_gs33_v4.yaml ^
    --resume <path\to\best.pt>
```

What to watch in the log:

| Column   | Good sign |
|----------|-----------|
| `bce`    | Decreases epoch 1 → end |
| `cls_px` | Decreases from ~1.5 toward <0.8 |
| `val cls_px` | Tracks train; large gap = overfitting → use `best.pt` |
| `dist`   | Decreases toward <1.0 |


In [ ]:
if RUN_TRAINING:
    cfg_path = MAIN_CONFIG if RUN_STAGE == "main" else FINETUNE_CONFIG
    rc = run_training(
        config_path       = cfg_path,
        repo_root         = _REPO_ROOT,
        resume_checkpoint = RESUME_CHECKPOINT,
        resume_strict     = RESUME_STRICT,
    )
    assert rc == 0, f"Training failed (exit {rc})"
else:
    print("RUN_TRAINING=False — configs written, launch from terminal:")
    print(f"  python -m shared_convnext_stardist_decoder.train_v2 --config {MAIN_CONFIG}")
    print(f"  python -m shared_convnext_stardist_decoder.train_v2 --config {FINETUNE_CONFIG} --resume <best.pt>")
